In [1]:
pip install guppylang

Note: you may need to restart the kernel to use updated packages.


In [2]:
from guppylang import guppy
from guppylang.std.quantum import qubit, h, rx, rz, cx, measure, angle
from collections import Counter
import time

In [3]:
@guppy
def zz_rotation(qi: qubit, qj: qubit, gamma: float) -> None:
    cx(qi, qj)
    rz(qj, angle(2.0 * gamma))
    cx(qi, qj)


@guppy
def cost_layer_3(q0: qubit, q1: qubit, q2: qubit, gamma: float) -> None:
    zz_rotation(q0, q1, gamma)
    zz_rotation(q1, q2, gamma)
    zz_rotation(q0, q2, gamma)


@guppy
def mixer_layer_3(q0: qubit, q1: qubit, q2: qubit, beta: float) -> None:
    rx(q0, angle(2.0 * beta))
    rx(q1, angle(2.0 * beta))
    rx(q2, angle(2.0 * beta))


@guppy
def qaoa_3() -> None:               
    gamma = 0.8                     
    beta  = 0.6             

    q0 = qubit()
    q1 = qubit()
    q2 = qubit()

    h(q0); h(q1); h(q2)

    cost_layer_3(q0, q1, q2, gamma)
    mixer_layer_3(q0, q1, q2, beta)

    r0 = measure(q0); result("q0", r0)
    r1 = measure(q1); result("q1", r1)
    r2 = measure(q2); result("q2", r2)


In [4]:
gamma = 0.8
beta = 0.4
shots = 512

bits = []

sim = qaoa_3.emulator(n_qubits=3).statevector_sim()

start = time.time()

for i in range(shots):
    r = sim.with_seed(i).run()
    d = dict(r.results[0].entries)
    bitstring = f"{d['q0']}{d['q1']}{d['q2']}"
    bits.append(bitstring)

end = time.time()

counts = Counter(bits)
print("Counts:", counts)
print("Tiempo:", end - start)


Counts: Counter({'000': 74, '001': 73, '111': 72, '100': 67, '011': 63, '010': 63, '101': 54, '110': 46})
Tiempo: 36.74099040031433


In [5]:
def maxcut(bitstring):
    score = 0
    edges = [(0,1),(1,2),(0,2)]
    for i,j in edges:
        if bitstring[i] != bitstring[j]:
            score += 1
    return score

total = sum(counts.values())
exp_val = sum(maxcut(b) * c for b,c in counts.items()) / total
print("Valor esperado MaxCut:", exp_val)


Valor esperado MaxCut: 1.4296875
